In [2]:
!pip install -q scikit-learn SpeechRecognition pydub

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

commands = [
    "turn on fan",
    "switch on fan",
    "start the fan",
    "fan on",
    "please turn on the fan",
    "turn off fan",
    "switch off fan",
    "stop the fan",
    "fan off",
    "please turn off the fan",
    "turn on light",
    "switch on light",
    "light on",
    "turn off light",
    "switch off light",
    "light off",
    "turn on all",
    "turn on everything",
    "turn off all",
    "turn off everything"
]

labels = [
    "fan_on",
    "fan_on",
    "fan_on",
    "fan_on",
    "fan_on",
    "fan_off",
    "fan_off",
    "fan_off",
    "fan_off",
    "fan_off",
    "light_on",
    "light_on",
    "light_on",
    "light_off",
    "light_off",
    "light_off",
    "all_on",
    "all_on",
    "all_off",
    "all_off"
]
model = Pipeline([
    ("vectorizer", TfidfVectorizer()),
    ("classifier", MultinomialNB())
])

model.fit(commands, labels)

print("Machine learning model trained successfully!")

appliances = {
    "fan": False,
    "light": False
}

def show_status():
    print("\n--- SMART HOME STATUS ---")
    for name, state in appliances.items():
        status = "ON" if state else "OFF"
        print(f"{name.title()}: {status}")

from google.colab import output
from IPython.display import Javascript, display
import base64
import io
from pydub import AudioSegment
import speech_recognition as sr

def record_voice():
    display(Javascript("""
    async function recordAudio() {
      const stream = await navigator.mediaDevices.getUserMedia({
        audio: true
      });

      const recorder = new MediaRecorder(stream);
      const chunks = [];

      recorder.ondataavailable = e => chunks.push(e.data);

      const stopped = new Promise(resolve => {
        recorder.onstop = resolve;
      });

      recorder.start();

      const msg = document.createElement("p");
      msg.id = "recording_message";
      msg.innerText = "Recording... Speak your command now.";
      document.body.appendChild(msg);

      await new Promise(resolve => setTimeout(resolve, 5000));
      recorder.stop();
      await stopped;
      stream.getTracks().forEach(track => track.stop());

      msg.innerText = "Recording complete.";

      const blob = new Blob(chunks, {type: recorder.mimeType});
      const buffer = await blob.arrayBuffer();
      const bytes = new Uint8Array(buffer);
      let binary = "";
      for (let i = 0; i < bytes.length; i++) {
        binary += String.fromCharCode(bytes[i]);
      }
      return btoa(binary);
    }
    """))

    audio_b64 = output.eval_js("recordAudio()")
    audio_bytes = base64.b64decode(audio_b64)

    audio = AudioSegment.from_file(
        io.BytesIO(audio_bytes)
    ).set_channels(1).set_frame_rate(16000)

    wav_buffer = io.BytesIO()
    audio.export(wav_buffer, format="wav")
    wav_buffer.seek(0)

    recognizer = sr.Recognizer()
    with sr.AudioFile(wav_buffer) as source:
        recorded_audio = recognizer.record(source)

    return recognizer.recognize_google(recorded_audio)

def control_appliances(command):
    command = command.lower().strip()
    print("Recognized voice:", command)

    # Check ALL appliances first
    if ("all" in command or "everything" in command) and any(
        word in command for word in ["on", "start", "enable"]
    ):
        appliances["fan"] = True
        appliances["light"] = True

    elif ("all" in command or "everything" in command) and any(
        word in command for word in ["off", "stop", "disable"]
    ):
        appliances["fan"] = False
        appliances["light"] = False

    # Turn ON both appliances
    elif ("fan" in command and "light" in command) and "on" in command:
        appliances["fan"] = True
        appliances["light"] = True

    # Individual fan control
    elif "fan" in command and any(
        word in command for word in ["on", "start"]
    ):
        appliances["fan"] = True

    elif "fan" in command and any(
        word in command for word in ["off", "stop"]
    ):
        appliances["fan"] = False

    # Individual light control
    elif "light" in command and any(
        word in command for word in ["on", "start"]
    ):
        appliances["light"] = True

    elif "light" in command and any(
        word in command for word in ["off", "stop"]
    ):
        appliances["light"] = False

    else:
        print("Command not recognized. Please try again.")
        return

    show_status()


def run_voice_command():
    try:
        print("Speak within the next 5 seconds...")
        command = record_voice()
        control_appliances(command)

    except sr.UnknownValueError:
        print("Sorry, I could not understand the voice.")
    except sr.RequestError as e:
        print("Speech recognition service error:", e)
    except Exception as e:
        print("Error:", e)

run_voice_command()

Machine learning model trained successfully!
Speak within the next 5 seconds...


<IPython.core.display.Javascript object>

Recognized voice: turn on light

--- SMART HOME STATUS ---
Fan: OFF
Light: ON
